In [1]:
import os
proj_dir = r"C:\Users\saioa.DESKTOP-NE4E1ID\miniforge3\envs\ect313\Library\share\proj"
assert os.path.exists(os.path.join(proj_dir, "proj.db")), "proj.db not found here"
os.environ["PROJ_DATA"] = proj_dir
os.environ["PROJ_LIB"] = proj_dir

import pyproj
pyproj.datadir.set_data_dir(proj_dir)
print(pyproj.datadir.get_data_dir())
pyproj.CRS("OGC:CRS84")  # should not raise

C:\Users\saioa.DESKTOP-NE4E1ID\miniforge3\envs\ect313\Library\share\proj


<Geographic 2D CRS: OGC:CRS84>
Name: WGS 84 (CRS84)
Axis Info [ellipsoidal]:
- Lon[east]: Geodetic longitude (degree)
- Lat[north]: Geodetic latitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [3]:
from xcube.core.store import new_data_store
from esa_climate_toolbox.core import get_op
from esa_climate_toolbox.core import list_ecv_datasets
from esa_climate_toolbox.core import get_store
from esa_climate_toolbox.core import list_datasets
from esa_climate_toolbox.ops import plot
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import cartopy.crs as ccrs
import warnings
warnings.filterwarnings("ignore") # Suppressing warnings to keep notebook output clean
%matplotlib inline

In [4]:
## Step 2: Connect to the ESA CCI Data Store

cci_store = new_data_store("esa-cci")
# List all available data sets of an ECV
list_ecv_datasets("LC")

[('esacci.LC.13-yrs.L4.WB.ASAR.Envisat.Map.4-0.r1', 'esa-cci'),
 ('esacci.LC.yr.L4.LCCS.multi-sensor.multi-platform.Map.2-0-7.r1', 'esa-cci'),
 ('esacci.LC.yr.L4.PFT.Unspecified.Unspecified.Map.2-0-81.r1', 'esa-cci'),
 ('esacci.LC.5-yrs.L4.CHANGE.multi-sensor.multi-platform.HRLCC30-A01.v1-2.Africa',
  'esa-cci'),
 ('esacci.LC.5-yrs.L4.CHANGE.multi-sensor.multi-platform.HRLCC30-A02.v1-2.Amazonia',
  'esa-cci'),
 ('esacci.LC.5-yrs.L4.CHANGE.multi-sensor.multi-platform.HRLCC30-A03.v1-2.Siberia',
  'esa-cci'),
 ('esacci.LC.5-yrs.L4.Map.multi-sensor.multi-platform.HRLC30-A01.v1-2.Africa',
  'esa-cci'),
 ('esacci.LC.5-yrs.L4.Map.multi-sensor.multi-platform.HRLC30-A02.v1-2.Amazonia',
  'esa-cci'),
 ('esacci.LC.5-yrs.L4.Map.multi-sensor.multi-platform.HRLC30-A03.v1-2.Siberia',
  'esa-cci'),
 ('esacci.LC.yr.L4.Map.multi-sensor.multi-platform.HRLC10-A01.v1-2.Africa',
  'esa-cci'),
 ('esacci.LC.yr.L4.Map.multi-sensor.multi-platform.HRLC10-A02.v1-2.Amazonia',
  'esa-cci'),
 ('esacci.LC.yr.L4.Map.m

In [4]:
## Step 3: Define the Dataset ID

data_id = 'esacci.LC.yr.L4.LCCS.multi-sensor.multi-platform.Map.2-0-7.r1'

In [5]:
## Step 6: Define Region, Time Range, and Variables of Interest

variables = ['lccs_class']  # Variable to retrieve
start_date = '2017-08-01'    # Start and end date (same for a single timestep)
end_date = '2017-08-01'
# N: 49.182407, 
# S: 49.001548,
# W: -114.365292
# E: -113.780778
bbox = (-114.50, 48.50, -113.50, 49.50)

# bbox = (-114.00, 49.00, -113.90, 49.01)

In [6]:
## Step 7: Open the dataset

land_cover_ds = cci_store.open_data(
    data_id=data_id,
    variable_names=variables,
    time_range=[start_date, end_date]
)

In [7]:
print("\nOpened Dataset:\n", land_cover_ds)


Opened Dataset:
 <xarray.Dataset> Size: 34GB
Dimensions:     (time: 1, lat: 64800, lon: 129600, bnds: 2)
Coordinates:
  * time        (time) datetime64[ns] 8B 2017-07-02T12:00:00
  * lat         (lat) float32 259kB 90.0 90.0 89.99 89.99 ... -89.99 -90.0 -90.0
  * lon         (lon) float32 518kB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
    time_bnds   (time, bnds) datetime64[ns] 16B dask.array<chunksize=(1, 2), meta=np.ndarray>
Dimensions without coordinates: bnds
Data variables:
    lccs_class  (time, lat, lon) float32 34GB dask.array<chunksize=(1, 2025, 2025), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    title:                   esacci.LC.yr.L4.LCCS.multi-sensor.multi-platform...
    date_created:            2026-10-16T17:26:08.962701
    processing_level:        L4
    time_coverage_start:     2017-01-01T00:00:00
    time_coverage_end:       2018-01-01T00:00:00
    time_coverage_duration:  P365DT0H0M0S
    history:                 [{'program': 'xcube_cci.ch

In [ ]:
# Reduce dimensions drastically — just wildfirearea
small_subset = land_cover_ds["lccs_class"].isel(time=0).sel(
    lat=slice(49.5, 48.5), 
    lon=slice(-114.5, -113.5)
    )

# Trigger loading only the small area
small_subset_loaded = small_subset.load()

: 

In [ ]:
# Define some simplified example LCCS classes (you can expand as needed)
class_ids = [10, 50, 100, 130, 150, 160, 190, 200, 210, 220]
class_names = [
    "Cropland",
    "Forest", 
    "Shrubland",
    "Grassland & Herbaceous",
    "Sparse Vegetation",
    "Wetlands",
    "Urban",
    "Bare Areas",
    "Water",
    "Snow/Ice"
]
colors = [
    "#ffff64", "#006400", "#8ca000", "#ffb432", "#ffebaf",
    "#00785a", "#c31400", "#fff5d7", "#0046c8", "#ffffff"
]

# Create colormap and normalization
cmap = ListedColormap(colors)
norm = BoundaryNorm(class_ids + [max(class_ids)+1], ncolors=cmap.N)

# Reduce dimensions drastically — just wildfirearea
small_subset = land_cover_ds["lccs_class"].isel(time=0).sel(
    lat=slice(49.5, 48.5), 
    lon=slice(-114.5, -113.5)
    )

# Trigger loading only the small area
small_subset_loaded = small_subset.load()

# Get date for the title
time_str = land_cover_ds['time'].isel(time=0).dt.strftime('%d %B %Y').item()

# Plot
plt.figure(figsize=(10, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
img = small_subset_loaded.plot(
    ax=ax,
    cmap=cmap,
    norm=norm,
    transform=ccrs.PlateCarree(),
    add_colorbar=False
)
ax.coastlines()
ax.set_title("Land Cover Classification – " + time_str)

# Add custom legend
from matplotlib.patches import Patch
legend_handles = [Patch(color=colors[i], label=class_names[i]) for i in range(len(class_ids))]
plt.legend(handles=legend_handles, bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()